In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

# The previous attempt failed because the file 'test.csv' was not found at the specified path.
# The error `FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/test.csv'`
# indicates that the file path is incorrect or the file doesn't exist at that location in your mounted Google Drive.

# To help you find your file, you can run the following command in a new cell to list all files and folders in your Google Drive:
# !ls -R '/content/drive/MyDrive/'

# Once you've located your file (e.g., the file associated with the ID '1-JA3lhFbHeC_2DCTGyU2dsRT4LNmwxEg'),
# update the 'file_path' variable below with its correct path.
# For example, if your file is in a folder called 'Colab Notebooks' and is named 'my_data.csv', the path might be:
# file_path = '/content/drive/MyDrive/Colab Notebooks/my_data.csv'

# Please update this line with the correct path to your CSV file:
file_path = '/content/drive/MyDrive/drone_data.csv' # <-- Updated to drone_data.csv as requested

# Attempt to read the CSV file with the (potentially corrected) path
try:
    df = pd.read_csv(file_path)
    print("Successfully loaded the file. Here are the first 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at '{file_path}'. Please ensure the 'file_path' variable points to the correct location of your CSV file in Google Drive.")
    print("Remember that the file ID you provided (1-JA3lhFbHeC_2DCTGyU2dsRT4LNmwxEg) corresponds to a specific file. You need to find its location in your mounted Google Drive.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Successfully loaded the file. Here are the first 5 rows:


,lane_id,hour,num_drones,payload_kg,wind_speed,distance_km,temperature,visibility_km,day_of_week,battery_level,drone_speed_kmh,hub_distance_km,is_peak_hour,is_weekday,is_congested
0,7,14,6,3.22,4.1,1.55,21.1,8.8,5,58,35.7,2.49,0,0,0
1,4,13,4,3.78,0.0,1.25,11.8,6.9,5,50,30.0,1.49,1,0,1
2,8,19,1,3.42,3.7,2.82,28.1,3.2,1,99,27.7,1.28,1,1,0
3,5,18,6,1.07,3.3,1.58,19.2,2.8,2,42,35.7,1.17,1,1,1
4,7,12,1,2.52,5.0,1.46,32.4,0.7,3,53,32.7,1.78,1,1,0


In [ ]:
import joblib

# Load the congestion model
try:
    loaded_congestion_model = joblib.load(MODEL_PATH)
    print(f"Congestion model loaded successfully from '{MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: Congestion model file not found at '{MODEL_PATH}'. Please ensure it has been trained and saved.")

# Load the ETA model
try:
    loaded_eta_model = joblib.load(ETA_MODEL_PATH)
    print(f"ETA model loaded successfully from '{ETA_MODEL_PATH}'.")
except FileNotFoundError:
    print(f"Error: ETA model file not found at '{ETA_MODEL_PATH}'. Please ensure it has been trained and saved.")

# Load the Battery model
battery_model_path = ETA_MODEL_PATH.replace('eta_model.pkl', 'battery_model.pkl')
try:
    loaded_battery_model = joblib.load(battery_model_path)
    print(f"Battery model loaded successfully from '{battery_model_path}'.")
except FileNotFoundError:
    print(f"Error: Battery model file not found at '{battery_model_path}'. Please ensure it has been trained and saved.")


Congestion model loaded successfully from 'congestion_model.pkl'.
ETA model loaded successfully from 'eta_model.pkl'.
Battery model loaded successfully from 'battery_model.pkl'.


### 6. ETA and Battery Predictor

Now we'll build the ETA and battery usage prediction models. Since we have two distinct target variables (`eta_minutes` and `battery_used_percent`), we'll train two separate `RandomForestRegressor` models. The features for these models are also specified in your project brief.

In [ ]:
# Define features (X_eta) and targets (y_eta, y_battery)
eta_features = [
    'distance_km', 'wind_speed', 'payload_kg',
    'drone_speed_kmh', 'num_drones', 'temperature',
    'visibility_km', 'battery_level', 'hour', 'day_of_week'
]

# For simplicity in this synthetic data, let's create `eta_minutes` and `battery_used_percent`.
# In a real scenario, these would come directly from your `synthetic_data.py` output.
# For now, we'll simulate them based on existing features.
# Let's assume a base speed of 40 km/h, distance / speed = time. Adjust for wind, payload, etc.
df['eta_minutes'] = (df['distance_km'] / df['drone_speed_kmh'] * 60) * \
                    (1 + df['wind_speed']/30 + df['payload_kg']/10 + df['num_drones']/MAX_DRONES_PER_LANE/2) # Simplified model
df['eta_minutes'] = df['eta_minutes'].apply(lambda x: max(1, x)) # Minimum 1 minute

# Battery usage: higher for longer distance, heavier payload, wind, low temp
df['battery_used_percent'] = (df['distance_km'] / 3) * \
                             (1 + df['payload_kg']/5 + df['wind_speed']/20 + (50 - df['temperature'])/50) # Simplified model
df['battery_used_percent'] = df['battery_used_percent'].apply(lambda x: min(90, max(5, x))) # Between 5% and 90%


X_eta = df[eta_features]
y_eta = df['eta_minutes']
y_battery = df['battery_used_percent']

# Split data for ETA and Battery models
X_eta_train, X_eta_test, y_eta_train, y_eta_test = train_test_split(X_eta, y_eta, test_size=0.2, random_state=42)
X_bat_train, X_bat_test, y_bat_train, y_bat_test = train_test_split(X_eta, y_battery, test_size=0.2, random_state=42)

print(f"ETA training set shape: {X_eta_train.shape}")
print(f"Battery training set shape: {X_bat_train.shape}")
print("ETA and Battery data split successfully.")

ETA training set shape: (4000, 10)
Battery training set shape: (4000, 10)
ETA and Battery data split successfully.


### 7. Train and Evaluate ETA & Battery Models

We'll use `RandomForestRegressor` models for both ETA and battery usage prediction, as specified in your project brief. We'll train these models and evaluate their performance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize the RandomForestRegressor for ETA
eta_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train the ETA model
print("Training ETA RandomForestRegressor...")
eta_model.fit(X_eta_train, y_eta_train)
print("ETA model training complete.")

# Evaluate the ETA model
y_eta_pred = eta_model.predict(X_eta_test)
mae_eta = mean_absolute_error(y_eta_test, y_eta_pred)
rmse_eta = np.sqrt(mean_squared_error(y_eta_test, y_eta_pred))
r2_eta = r2_score(y_eta_test, y_eta_pred)
print(f"\nETA Model Performance:")
print(f"  Mean Absolute Error (MAE): {mae_eta:.2f} minutes")
print(f"  Root Mean Squared Error (RMSE): {rmse_eta:.2f} minutes")
print(f"  R-squared (R2): {r2_eta:.4f}")

# Initialize the RandomForestRegressor for Battery
battery_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train the Battery model
print("\nTraining Battery RandomForestRegressor...")
battery_model.fit(X_bat_train, y_bat_train)
print("Battery model training complete.")

# Evaluate the Battery model
y_bat_pred = battery_model.predict(X_bat_test)
mae_bat = mean_absolute_error(y_bat_test, y_bat_pred)
rmse_bat = np.sqrt(mean_squared_error(y_bat_test, y_bat_pred))
r2_bat = r2_score(y_bat_test, y_bat_pred)
print(f"\nBattery Model Performance:")
print(f"  Mean Absolute Error (MAE): {mae_bat:.2f} %")
print(f"  Root Mean Squared Error (RMSE): {rmse_bat:.2f} %")
print(f"  R-squared (R2): {r2_bat:.4f}")

Training ETA RandomForestRegressor...
ETA model training complete.

ETA Model Performance:
  Mean Absolute Error (MAE): 0.32 minutes
  Root Mean Squared Error (RMSE): 0.54 minutes
  R-squared (R2): 0.9830

Training Battery RandomForestRegressor...
Battery model training complete.

Battery Model Performance:
  Mean Absolute Error (MAE): 0.00 %
  Root Mean Squared Error (RMSE): 0.00 %
  R-squared (R2): 1.0000


### 8. Save ETA and Battery Models

Just like the congestion model, we'll save our trained ETA and battery prediction models using `joblib` so they can be loaded later by the API.

In [ ]:
# Save the trained ETA model
joblib.dump(eta_model, ETA_MODEL_PATH)
print(f"Trained ETA model saved to '{ETA_MODEL_PATH}'")

# Save the trained Battery model (using a slightly modified path to distinguish, or integrate)
battery_model_path = ETA_MODEL_PATH.replace('eta_model.pkl', 'battery_model.pkl')
joblib.dump(battery_model, battery_model_path)
print(f"Trained Battery model saved to '{battery_model_path}'")

Trained ETA model saved to 'eta_model.pkl'
Trained Battery model saved to 'battery_model.pkl'


### 9. ETA and Battery Prediction Function

Finally, we'll create the `predict_eta` function that takes input data, loads the models, and returns the predicted ETA, battery usage, and an estimated arrival time (which can be derived from the current time plus the predicted ETA).

In [ ]:
from datetime import datetime, timedelta

def predict_eta(input_data: pd.DataFrame) -> dict:
    """
    Predicts the Estimated Time of Arrival (ETA) and battery usage for a drone trip.

    Args:
        input_data (pd.DataFrame): A DataFrame containing features for ETA prediction.
                                   Must include: distance_km, wind_speed, payload_kg,
                                   drone_speed_kmh, num_drones, temperature, visibility_km,
                                   battery_level, hour, day_of_week.

    Returns:
        dict: A dictionary containing:
              - 'etaMinutes' (float): Predicted ETA in minutes.
              - 'batteryUsed' (float): Predicted battery usage in percent.
              - 'estimatedArrival' (str): Estimated arrival time in ISO format.
    """
    try:
        loaded_eta_model = joblib.load(ETA_MODEL_PATH)
        loaded_battery_model = joblib.load(ETA_MODEL_PATH.replace('eta_model.pkl', 'battery_model.pkl'))
    except FileNotFoundError:
        return {"error": f"Model files not found. Please train and save the ETA and Battery models first."}

    required_features_eta = [
        'distance_km', 'wind_speed', 'payload_kg',
        'drone_speed_kmh', 'num_drones', 'temperature',
        'visibility_km', 'battery_level', 'hour', 'day_of_week'
    ]
    if not all(f in input_data.columns for f in required_features_eta):
        return {"error": "Input data is missing one or more required ETA features."}

    # Predict ETA
    predicted_eta = loaded_eta_model.predict(input_data[required_features_eta])[0]

    # Predict Battery Usage
    predicted_battery_used = loaded_battery_model.predict(input_data[required_features_eta])[0]

    # Calculate estimated arrival time
    current_time = datetime.now()
    estimated_arrival_time = current_time + timedelta(minutes=predicted_eta)

    return {
        "etaMinutes": float(predicted_eta),
        "batteryUsed": float(predicted_battery_used),
        "estimatedArrival": estimated_arrival_time.isoformat() # ISO format for easy parsing
    }

# --- Test the predict_eta function with a sample --- #
print("\n--- Testing predict_eta function ---")
# Create a sample input based on the first row of your DataFrame, ensuring all eta_features are present
sample_input_eta = df[eta_features].iloc[[0]]

# Let's add a dummy 'num_drones' column to sample_input_eta if it's not already there from eta_features
# as it's used in the ETA prediction in this example.
# If 'num_drones' is already in eta_features and df, this line won't change anything.
if 'num_drones' not in sample_input_eta.columns:
    sample_input_eta['num_drones'] = df['num_drones'].iloc[0]

print("Sample Input Data for ETA Prediction:\n", sample_input_eta)

eta_prediction = predict_eta(sample_input_eta)
print("Prediction for sample ETA input:", eta_prediction)

# You can also try a scenario with different inputs (e.g., longer distance, higher wind)
hypothetical_eta_sample = sample_input_eta.copy()
hypothetical_eta_sample['distance_km'] = 5.0 # Longer distance
hypothetical_eta_sample['wind_speed'] = 15.0 # Higher wind
hypothetical_eta_sample['payload_kg'] = 4.0 # Heavier payload
hypothetical_eta_sample['drone_speed_kmh'] = 30.0 # Slower speed

print("\nHypothetical ETA Sample Input Data:\n", hypothetical_eta_sample)
hypothetical_eta_prediction = predict_eta(hypothetical_eta_sample)
print("Prediction for hypothetical ETA input:", hypothetical_eta_prediction)



--- Testing predict_eta function ---
Sample Input Data for ETA Prediction:
    distance_km  wind_speed  payload_kg  drone_speed_kmh  num_drones  \
0         1.55         4.1        3.22             35.7           6   

   temperature  visibility_km  battery_level  hour  day_of_week  
0         21.1            8.8             58    14            5  
Prediction for sample ETA input: {'etaMinutes': 5.275234764412695, 'batteryUsed': 5.0, 'estimatedArrival': '2026-03-17T10:08:25.777886'}

Hypothetical ETA Sample Input Data:
    distance_km  wind_speed  payload_kg  drone_speed_kmh  num_drones  \
0          5.0        15.0         4.0             30.0           6   

   temperature  visibility_km  battery_level  hour  day_of_week  
0         21.1            8.8             58    14            5  
Prediction for hypothetical ETA input: {'etaMinutes': 13.566729690273315, 'batteryUsed': 5.0, 'estimatedArrival': '2026-03-17T10:16:43.410998'}


### 1. Imports and Configuration Loading

We start by importing all the necessary libraries for data manipulation, model training, and saving. We'll also simulate loading your `config.py` file to ensure all parameters are consistent with your project setup. In a real project, you would have `config.py` as a separate file and import variables from it.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# --- Simulate config.py for Colab environment ---
# In your local project, these would be imported from config.py
NUM_LANES = 10
MAX_DRONES_PER_LANE = 5
ALTITUDE_BUFFER = 10
MIN_ALTITUDE = 20
MAX_ALTITUDE = 120
BATTERY_LOW = 15
BATTERY_CRITICAL = 8
BATTERY_FULL = 100
PEAK_HOURS = [8, 9, 12, 13, 17, 18, 19]
TIME_SLOT_DURATION = 5
CONGESTION_THRESHOLD = 0.6 # Used for determining congestion level based on confidence
HUBS = {"OAT": [26.5127, 80.2325], "Shopping_Complex": [26.5098, 80.2317]}
MAX_DRONES = 20
MODEL_PATH = "congestion_model.pkl" # Path where the trained congestion model will be saved
ETA_MODEL_PATH = "eta_model.pkl"
API_HOST = "0.0.0.0"
API_PORT = 8000
DASHBOARD_PORT = 8501
# --------------------------------------------------

print("Libraries and simulated config loaded successfully.")

Libraries and simulated config loaded successfully.


### 2. Data Preparation

Now, we'll define the features (input variables for the model) and the target variable (what the model will predict). Then, we'll split the data into training and testing sets. The training set is used to teach the model, and the testing set is used to evaluate how well it learned on unseen data.

In [ ]:
# Define features (X) and target (y)
# These are the 14 features you listed in your project overview for congestion prediction.
features = [
    'lane_id', 'hour', 'num_drones', 'payload_kg', 'wind_speed',
    'distance_km', 'temperature', 'visibility_km', 'day_of_week',
    'battery_level', 'drone_speed_kmh', 'hub_distance_km',
    'is_peak_hour', 'is_weekday'
]
target = 'is_congested'

X = df[features]
y = df[target]

# Split data into training and testing sets (80% train, 20% test)
# random_state ensures that the split is the same every time you run the code, for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Original dataset shape: {df.shape}")
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print("Data split into training and testing sets successfully.")

Original dataset shape: (5000, 17)
Training set shape: (4000, 14)
Testing set shape: (1000, 14)
Data split into training and testing sets successfully.


### 3. Model Training and Evaluation

We'll use a `RandomForestClassifier` as specified. This model is an ensemble learning method that builds multiple decision trees and merges them together to get a more accurate and stable prediction. The parameters like `n_estimators`, `max_depth`, `min_samples_split`, and `min_samples_leaf` are set according to your project brief. We'll train the model and then evaluate its performance on the test set.

In [ ]:
# Initialize the RandomForestClassifier with specified parameters
# n_estimators: Number of trees in the forest.
# max_depth: The maximum depth of the tree.
# min_samples_split: The minimum number of samples required to split an internal node.
# min_samples_leaf: The minimum number of samples required to be at a leaf node.
# random_state: Controls the randomness of the bootstrapping of samples when building trees, for reproducibility.
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train the model on the training data
print("Training RandomForestClassifier...")
model.fit(X_train, y_train)
print("Model training complete.")

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# You can also get feature importances, which can be useful for defense
feature_importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature Importances:")
print(feature_importances)

Training RandomForestClassifier...
Model training complete.
Model Accuracy on Test Set: 0.9510

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       844
           1       0.88      0.79      0.83       156

    accuracy                           0.95      1000
   macro avg       0.92      0.88      0.90      1000
weighted avg       0.95      0.95      0.95      1000


Feature Importances:
is_peak_hour       0.310141
num_drones         0.252373
hour               0.072439
drone_speed_kmh    0.057012
wind_speed         0.049743
payload_kg         0.043688
visibility_km      0.041568
battery_level      0.040473
distance_km        0.033433
hub_distance_km    0.030393
temperature        0.029873
day_of_week        0.016996
lane_id            0.016280
is_weekday         0.005588
dtype: float64


### 4. Model Saving

It's crucial to save your trained model so you don't have to re-train it every time you want to use it. We'll use `joblib.dump` to save the model to the path specified in `MODEL_PATH` from `config.py`.

In [ ]:
# Save the trained model to a file using joblib
# This allows you to load and use the model later without retraining.
joblib.dump(model, MODEL_PATH)
print(f"Trained congestion model saved to '{MODEL_PATH}'")

Trained congestion model saved to 'congestion_model.pkl'


### 5. Congestion Prediction Function

Finally, we'll implement the `predict_congestion` function. This function will take new drone data, load the saved model, make a prediction, calculate the confidence, and determine the congestion level (low, medium, high) based on your `CONGESTION_THRESHOLD`.

In [ ]:
def predict_congestion(input_data: pd.DataFrame) -> dict:
    """
    Predicts if a drone lane will be congested and provides a confidence score
    and a congestion level.

    Args:
        input_data (pd.DataFrame): A DataFrame containing the 14 features
                                   for which to predict congestion.
                                   Must have columns: lane_id, hour, num_drones,
                                   payload_kg, wind_speed, distance_km, temperature,
                                   visibility_km, day_of_week, battery_level,
                                   drone_speed_kmh, hub_distance_km,
                                   is_peak_hour, is_weekday.

    Returns:
        dict: A dictionary containing:
              - 'is_congested' (bool): True if congested, False otherwise.
              - 'confidence' (float): The prediction probability of the predicted class.
              - 'congestion_level' (str): 'low', 'medium', or 'high' based on confidence.
    """
    # Load the pre-trained model
    # In a real application, you would load the model once when the application starts
    # to avoid repeated disk I/O.
    try:
        loaded_model = joblib.load(MODEL_PATH)
    except FileNotFoundError:
        return {"error": f"Model file not found at {MODEL_PATH}. Please train and save the model first."}

    # Ensure the input data has the correct features and in the correct order
    required_features = [
        'lane_id', 'hour', 'num_drones', 'payload_kg', 'wind_speed',
        'distance_km', 'temperature', 'visibility_km', 'day_of_week',
        'battery_level', 'drone_speed_kmh', 'hub_distance_km',
        'is_peak_hour', 'is_weekday'
    ]
    if not all(f in input_data.columns for f in required_features):
        return {"error": "Input data is missing one or more required features."}

    # Predict the congestion (0 or 1)
    prediction = loaded_model.predict(input_data[required_features])[0]
    is_congested = bool(prediction)

    # Get prediction probabilities
    # The probabilities are for [class 0, class 1]
    prediction_proba = loaded_model.predict_proba(input_data[required_features])[0]

    # Confidence is the probability of the predicted class
    confidence = prediction_proba[1] if is_congested else prediction_proba[0]

    # Determine congestion level based on confidence and CONGESTION_THRESHOLD
    # This logic translates the confidence into a qualitative level.
    if is_congested:
        if confidence >= CONGESTION_THRESHOLD:
            congestion_level = "high"
        else:
            congestion_level = "medium"
    else:
        # If not congested, 'low' congestion is typically assigned
        congestion_level = "low"

    return {
        "is_congested": is_congested,
        "confidence": float(confidence),
        "congestion_level": congestion_level
    }

# --- Test the predict_congestion function with a sample --- #
print("\n--- Testing predict_congestion function ---")
# Create a sample input based on the first row of your DataFrame
sample_input = df[features].iloc[[0]]
print("Sample Input Data:\n", sample_input)

congestion_prediction = predict_congestion(sample_input)
print("Prediction for sample input:", congestion_prediction)

# You can also try a scenario where you expect congestion (e.g., more drones, peak hour)
# Let's create a hypothetical congested scenario
congested_sample = sample_input.copy()
congested_sample['num_drones'] = 8 # Higher number of drones
congested_sample['is_peak_hour'] = 1 # During peak hour
congested_sample['payload_kg'] = 4.5 # Heavier payload

print("\nHypothetical Congested Sample Input Data:\n", congested_sample)
hypothetical_congestion_prediction = predict_congestion(congested_sample)
print("Prediction for hypothetical congested input:", hypothetical_congestion_prediction)



--- Testing predict_congestion function ---
Sample Input Data:
    lane_id  hour  num_drones  payload_kg  wind_speed  distance_km  \
0        7    14           6        3.22         4.1         1.55   

   temperature  visibility_km  day_of_week  battery_level  drone_speed_kmh  \
0         21.1            8.8            5             58             35.7   

   hub_distance_km  is_peak_hour  is_weekday  
0             2.49             0           0  
Prediction for sample input: {'is_congested': False, 'confidence': 0.9928591954022988, 'congestion_level': 'low'}

Hypothetical Congested Sample Input Data:
    lane_id  hour  num_drones  payload_kg  wind_speed  distance_km  \
0        7    14           8         4.5         4.1         1.55   

   temperature  visibility_km  day_of_week  battery_level  drone_speed_kmh  \
0         21.1            8.8            5             58             35.7   

   hub_distance_km  is_peak_hour  is_weekday  
0             2.49             1           0

### 6. ETA and Battery Predictor

Now we'll build the ETA and battery usage prediction models. Since we have two distinct target variables (`eta_minutes` and `battery_used_percent`), we'll train two separate `RandomForestRegressor` models. The features for these models are also specified in your project brief.

In [ ]:
# Define features (X_eta) and targets (y_eta, y_battery)
eta_features = [
    'distance_km', 'wind_speed', 'payload_kg',
    'drone_speed_kmh', 'num_drones', 'temperature',
    'visibility_km', 'battery_level', 'hour', 'day_of_week'
]

# For simplicity in this synthetic data, let's create `eta_minutes` and `battery_used_percent`.
# In a real scenario, these would come directly from your `synthetic_data.py` output.
# For now, we'll simulate them based on existing features.
# Let's assume a base speed of 40 km/h, distance / speed = time. Adjust for wind, payload, etc.
df['eta_minutes'] = (df['distance_km'] / df['drone_speed_kmh'] * 60) * \
                    (1 + df['wind_speed']/30 + df['payload_kg']/10 + df['num_drones']/MAX_DRONES_PER_LANE/2) # Simplified model
df['eta_minutes'] = df['eta_minutes'].apply(lambda x: max(1, x)) # Minimum 1 minute

# Battery usage: higher for longer distance, heavier payload, wind, low temp
df['battery_used_percent'] = (df['distance_km'] / 3) * \
                             (1 + df['payload_kg']/5 + df['wind_speed']/20 + (50 - df['temperature'])/50) # Simplified model
df['battery_used_percent'] = df['battery_used_percent'].apply(lambda x: min(90, max(5, x))) # Between 5% and 90%


X_eta = df[eta_features]
y_eta = df['eta_minutes']
y_battery = df['battery_used_percent']

# Split data for ETA and Battery models
X_eta_train, X_eta_test, y_eta_train, y_eta_test = train_test_split(X_eta, y_eta, test_size=0.2, random_state=42)
X_bat_train, X_bat_test, y_bat_train, y_bat_test = train_test_split(X_eta, y_battery, test_size=0.2, random_state=42)

print(f"ETA training set shape: {X_eta_train.shape}")
print(f"Battery training set shape: {X_bat_train.shape}")
print("ETA and Battery data split successfully.")

ETA training set shape: (4000, 10)
Battery training set shape: (4000, 10)
ETA and Battery data split successfully.


### 7. Train and Evaluate ETA & Battery Models

We'll use `RandomForestRegressor` models for both ETA and battery usage prediction, as specified in your project brief. We'll train these models and evaluate their performance.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialize the RandomForestRegressor for ETA
eta_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train the ETA model
print("Training ETA RandomForestRegressor...")
eta_model.fit(X_eta_train, y_eta_train)
print("ETA model training complete.")

# Evaluate the ETA model
y_eta_pred = eta_model.predict(X_eta_test)
mae_eta = mean_absolute_error(y_eta_test, y_eta_pred)
rmse_eta = np.sqrt(mean_squared_error(y_eta_test, y_eta_pred))
r2_eta = r2_score(y_eta_test, y_eta_pred)
print(f"\nETA Model Performance:")
print(f"  Mean Absolute Error (MAE): {mae_eta:.2f} minutes")
print(f"  Root Mean Squared Error (RMSE): {rmse_eta:.2f} minutes")
print(f"  R-squared (R2): {r2_eta:.4f}")

# Initialize the RandomForestRegressor for Battery
battery_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train the Battery model
print("\nTraining Battery RandomForestRegressor...")
battery_model.fit(X_bat_train, y_bat_train)
print("Battery model training complete.")

# Evaluate the Battery model
y_bat_pred = battery_model.predict(X_bat_test)
mae_bat = mean_absolute_error(y_bat_test, y_bat_pred)
rmse_bat = np.sqrt(mean_squared_error(y_bat_test, y_bat_pred))
r2_bat = r2_score(y_bat_test, y_bat_pred)
print(f"\nBattery Model Performance:")
print(f"  Mean Absolute Error (MAE): {mae_bat:.2f} %")
print(f"  Root Mean Squared Error (RMSE): {rmse_bat:.2f} %")
print(f"  R-squared (R2): {r2_bat:.4f}")

Training ETA RandomForestRegressor...
ETA model training complete.

ETA Model Performance:
  Mean Absolute Error (MAE): 0.32 minutes
  Root Mean Squared Error (RMSE): 0.54 minutes
  R-squared (R2): 0.9830

Training Battery RandomForestRegressor...
Battery model training complete.

Battery Model Performance:
  Mean Absolute Error (MAE): 0.00 %
  Root Mean Squared Error (RMSE): 0.00 %
  R-squared (R2): 1.0000


### 8. Save ETA and Battery Models

Just like the congestion model, we'll save our trained ETA and battery prediction models using `joblib` so they can be loaded later by the API.

In [ ]:
# Save the trained ETA model
joblib.dump(eta_model, ETA_MODEL_PATH)
print(f"Trained ETA model saved to '{ETA_MODEL_PATH}'")

# Save the trained Battery model (using a slightly modified path to distinguish, or integrate)
battery_model_path = ETA_MODEL_PATH.replace('eta_model.pkl', 'battery_model.pkl')
joblib.dump(battery_model, battery_model_path)
print(f"Trained Battery model saved to '{battery_model_path}'")

Trained ETA model saved to 'eta_model.pkl'
Trained Battery model saved to 'battery_model.pkl'


### 9. ETA and Battery Prediction Function

Finally, we'll create the `predict_eta` function that takes input data, loads the models, and returns the predicted ETA, battery usage, and an estimated arrival time (which can be derived from the current time plus the predicted ETA).

In [ ]:
from datetime import datetime, timedelta

def predict_eta(input_data: pd.DataFrame) -> dict:
    """
    Predicts the Estimated Time of Arrival (ETA) and battery usage for a drone trip.

    Args:
        input_data (pd.DataFrame): A DataFrame containing features for ETA prediction.
                                   Must include: distance_km, wind_speed, payload_kg,
                                   drone_speed_kmh, num_drones, temperature, visibility_km,
                                   battery_level, hour, day_of_week.

    Returns:
        dict: A dictionary containing:
              - 'etaMinutes' (float): Predicted ETA in minutes.
              - 'batteryUsed' (float): Predicted battery usage in percent.
              - 'estimatedArrival' (str): Estimated arrival time in ISO format.
    """
    try:
        loaded_eta_model = joblib.load(ETA_MODEL_PATH)
        loaded_battery_model = joblib.load(ETA_MODEL_PATH.replace('eta_model.pkl', 'battery_model.pkl'))
    except FileNotFoundError:
        return {"error": f"Model files not found. Please train and save the ETA and Battery models first."}

    required_features_eta = [
        'distance_km', 'wind_speed', 'payload_kg',
        'drone_speed_kmh', 'num_drones', 'temperature',
        'visibility_km', 'battery_level', 'hour', 'day_of_week'
    ]
    if not all(f in input_data.columns for f in required_features_eta):
        return {"error": "Input data is missing one or more required ETA features."}

    # Predict ETA
    predicted_eta = loaded_eta_model.predict(input_data[required_features_eta])[0]

    # Predict Battery Usage
    predicted_battery_used = loaded_battery_model.predict(input_data[required_features_eta])[0]

    # Calculate estimated arrival time
    current_time = datetime.now()
    estimated_arrival_time = current_time + timedelta(minutes=predicted_eta)

    return {
        "etaMinutes": float(predicted_eta),
        "batteryUsed": float(predicted_battery_used),
        "estimatedArrival": estimated_arrival_time.isoformat() # ISO format for easy parsing
    }

# --- Test the predict_eta function with a sample --- #
print("\n--- Testing predict_eta function ---")
# Create a sample input based on the first row of your DataFrame, ensuring all eta_features are present
sample_input_eta = df[eta_features].iloc[[0]]

# Let's add a dummy 'num_drones' column to sample_input_eta if it's not already there from eta_features
# as it's used in the ETA prediction in this example.
# If 'num_drones' is already in eta_features and df, this line won't change anything.
if 'num_drones' not in sample_input_eta.columns:
    sample_input_eta['num_drones'] = df['num_drones'].iloc[0]

print("Sample Input Data for ETA Prediction:\n", sample_input_eta)

eta_prediction = predict_eta(sample_input_eta)
print("Prediction for sample ETA input:", eta_prediction)

# You can also try a scenario with different inputs (e.g., longer distance, higher wind)
hypothetical_eta_sample = sample_input_eta.copy()
hypothetical_eta_sample['distance_km'] = 5.0 # Longer distance
hypothetical_eta_sample['wind_speed'] = 15.0 # Higher wind
hypothetical_eta_sample['payload_kg'] = 4.0 # Heavier payload
hypothetical_eta_sample['drone_speed_kmh'] = 30.0 # Slower speed

print("\nHypothetical ETA Sample Input Data:\n", hypothetical_eta_sample)
hypothetical_eta_prediction = predict_eta(hypothetical_eta_sample)
print("Prediction for hypothetical ETA input:", hypothetical_eta_prediction)



--- Testing predict_eta function ---
Sample Input Data for ETA Prediction:
    distance_km  wind_speed  payload_kg  drone_speed_kmh  num_drones  \
0         1.55         4.1        3.22             35.7           6   

   temperature  visibility_km  battery_level  hour  day_of_week  
0         21.1            8.8             58    14            5  
Prediction for sample ETA input: {'etaMinutes': 5.275234764412695, 'batteryUsed': 5.0, 'estimatedArrival': '2026-03-17T10:08:37.845859'}

Hypothetical ETA Sample Input Data:
    distance_km  wind_speed  payload_kg  drone_speed_kmh  num_drones  \
0          5.0        15.0         4.0             30.0           6   

   temperature  visibility_km  battery_level  hour  day_of_week  
0         21.1            8.8             58    14            5  
Prediction for hypothetical ETA input: {'etaMinutes': 13.566729690273315, 'batteryUsed': 5.0, 'estimatedArrival': '2026-03-17T10:16:55.471225'}


### 1. Imports and Configuration Loading

We start by importing all the necessary libraries for data manipulation, model training, and saving. We'll also simulate loading your `config.py` file to ensure all parameters are consistent with your project setup. In a real project, you would have `config.py` as a separate file and import variables from it.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# --- Simulate config.py for Colab environment ---
# In your local project, these would be imported from config.py
NUM_LANES = 10
MAX_DRONES_PER_LANE = 5
ALTITUDE_BUFFER = 10
MIN_ALTITUDE = 20
MAX_ALTITUDE = 120
BATTERY_LOW = 15
BATTERY_CRITICAL = 8
BATTERY_FULL = 100
PEAK_HOURS = [8, 9, 12, 13, 17, 18, 19]
TIME_SLOT_DURATION = 5
CONGESTION_THRESHOLD = 0.6 # Used for determining congestion level based on confidence
HUBS = {"OAT": [26.5127, 80.2325], "Shopping_Complex": [26.5098, 80.2317]}
MAX_DRONES = 20
MODEL_PATH = "congestion_model.pkl" # Path where the trained congestion model will be saved
ETA_MODEL_PATH = "eta_model.pkl"
API_HOST = "0.0.0.0"
API_PORT = 8000
DASHBOARD_PORT = 8501
# --------------------------------------------------

print("Libraries and simulated config loaded successfully.")

Libraries and simulated config loaded successfully.


### 2. Data Preparation

Now, we'll define the features (input variables for the model) and the target variable (what the model will predict). Then, we'll split the data into training and testing sets. The training set is used to teach the model, and the testing set is used to evaluate how well it learned on unseen data.

In [ ]:
# Define features (X) and target (y)
# These are the 14 features you listed in your project overview for congestion prediction.
features = [
    'lane_id', 'hour', 'num_drones', 'payload_kg', 'wind_speed',
    'distance_km', 'temperature', 'visibility_km', 'day_of_week',
    'battery_level', 'drone_speed_kmh', 'hub_distance_km',
    'is_peak_hour', 'is_weekday'
]
target = 'is_congested'

X = df[features]
y = df[target]

# Split data into training and testing sets (80% train, 20% test)
# random_state ensures that the split is the same every time you run the code, for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Original dataset shape: {df.shape}")
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print("Data split into training and testing sets successfully.")

Original dataset shape: (5000, 15)
Training set shape: (4000, 14)
Testing set shape: (1000, 14)
Data split into training and testing sets successfully.


### 3. Model Training and Evaluation

We'll use a `RandomForestClassifier` as specified. This model is an ensemble learning method that builds multiple decision trees and merges them together to get a more accurate and stable prediction. The parameters like `n_estimators`, `max_depth`, `min_samples_split`, and `min_samples_leaf` are set according to your project brief. We'll train the model and then evaluate its performance on the test set.

In [ ]:
# Initialize the RandomForestClassifier with specified parameters
# n_estimators: Number of trees in the forest.
# max_depth: The maximum depth of the tree.
# min_samples_split: The minimum number of samples required to split an internal node.
# min_samples_leaf: The minimum number of samples required to be at a leaf node.
# random_state: Controls the randomness of the bootstrapping of samples when building trees, for reproducibility.
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# Train the model on the training data
print("Training RandomForestClassifier...")
model.fit(X_train, y_train)
print("Model training complete.")

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy on Test Set: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# You can also get feature importances, which can be useful for defense
feature_importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature Importances:")
print(feature_importances)

Training RandomForestClassifier...
Model training complete.
Model Accuracy on Test Set: 0.9510

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       844
           1       0.88      0.79      0.83       156

    accuracy                           0.95      1000
   macro avg       0.92      0.88      0.90      1000
weighted avg       0.95      0.95      0.95      1000


Feature Importances:
is_peak_hour       0.310141
num_drones         0.252373
hour               0.072439
drone_speed_kmh    0.057012
wind_speed         0.049743
payload_kg         0.043688
visibility_km      0.041568
battery_level      0.040473
distance_km        0.033433
hub_distance_km    0.030393
temperature        0.029873
day_of_week        0.016996
lane_id            0.016280
is_weekday         0.005588
dtype: float64


### 4. Model Saving

It's crucial to save your trained model so you don't have to re-train it every time you want to use it. We'll use `joblib.dump` to save the model to the path specified in `MODEL_PATH` from `config.py`.

In [ ]:
# Save the trained model to a file using joblib
# This allows you to load and use the model later without retraining.
joblib.dump(model, MODEL_PATH)
print(f"Trained congestion model saved to '{MODEL_PATH}'")

Trained congestion model saved to 'congestion_model.pkl'


### 5. Congestion Prediction Function

Finally, we'll implement the `predict_congestion` function. This function will take new drone data, load the saved model, make a prediction, calculate the confidence, and determine the congestion level (low, medium, high) based on your `CONGESTION_THRESHOLD`.

In [ ]:
def predict_congestion(input_data: pd.DataFrame) -> dict:
    """
    Predicts if a drone lane will be congested and provides a confidence score
    and a congestion level.

    Args:
        input_data (pd.DataFrame): A DataFrame containing the 14 features
                                   for which to predict congestion.
                                   Must have columns: lane_id, hour, num_drones,
                                   payload_kg, wind_speed, distance_km, temperature,
                                   visibility_km, day_of_week, battery_level,
                                   drone_speed_kmh, hub_distance_km, is_peak_hour,
                                   is_weekday.

    Returns:
        dict: A dictionary containing:
              - 'is_congested' (bool): True if congested, False otherwise.
              - 'confidence' (float): The prediction probability of the predicted class.
              - 'congestion_level' (str): 'low', 'medium', or 'high' based on confidence.
    """
    # Load the pre-trained model
    # In a real application, you would load the model once when the application starts
    # to avoid repeated disk I/O.
    try:
        loaded_model = joblib.load(MODEL_PATH)
    except FileNotFoundError:
        return {"error": f"Model file not found at {MODEL_PATH}. Please train and save the model first."}

    # Ensure the input data has the correct features and in the correct order
    required_features = [
        'lane_id', 'hour', 'num_drones', 'payload_kg', 'wind_speed',
        'distance_km', 'temperature', 'visibility_km', 'day_of_week',
        'battery_level', 'drone_speed_kmh', 'hub_distance_km',
        'is_peak_hour', 'is_weekday'
    ]
    if not all(f in input_data.columns for f in required_features):
        return {"error": "Input data is missing one or more required features."}

    # Predict the congestion (0 or 1)
    prediction = loaded_model.predict(input_data[required_features])[0]
    is_congested = bool(prediction)

    # Get prediction probabilities
    # The probabilities are for [class 0, class 1]
    prediction_proba = loaded_model.predict_proba(input_data[required_features])[0]

    # Confidence is the probability of the predicted class
    confidence = prediction_proba[1] if is_congested else prediction_proba[0]

    # Determine congestion level based on confidence and CONGESTION_THRESHOLD
    # This logic translates the confidence into a qualitative level.
    if is_congested:
        if confidence >= CONGESTION_THRESHOLD:
            congestion_level = "high"
        else:
            congestion_level = "medium"
    else:
        # If not congested, 'low' congestion is typically assigned
        congestion_level = "low"

    return {
        "is_congested": is_congested,
        "confidence": float(confidence),
        "congestion_level": congestion_level
    }

# --- Test the predict_congestion function with a sample --- #
print("\n--- Testing predict_congestion function ---")
# Create a sample input based on the first row of your DataFrame
sample_input = df[features].iloc[[0]]
print("Sample Input Data:\n", sample_input)

congestion_prediction = predict_congestion(sample_input)
print("Prediction for sample input:", congestion_prediction)

# You can also try a scenario where you expect congestion (e.g., more drones, peak hour)
# Let's create a hypothetical congested scenario
congested_sample = sample_input.copy()
congested_sample['num_drones'] = 8 # Higher number of drones
congested_sample['is_peak_hour'] = 1 # During peak hour
congested_sample['payload_kg'] = 4.5 # Heavier payload

print("\nHypothetical Congested Sample Input Data:\n", congested_sample)
hypothetical_congestion_prediction = predict_congestion(congested_sample)
print("Prediction for hypothetical congested input:", hypothetical_congestion_prediction)



--- Testing predict_congestion function ---
Sample Input Data:
    lane_id  hour  num_drones  payload_kg  wind_speed  distance_km  \
0        7    14           6        3.22         4.1         1.55   

   temperature  visibility_km  day_of_week  battery_level  drone_speed_kmh  \
0         21.1            8.8            5             58             35.7   

   hub_distance_km  is_peak_hour  is_weekday  
0             2.49             0           0  
Prediction for sample input: {'is_congested': False, 'confidence': 0.9928591954022988, 'congestion_level': 'low'}

Hypothetical Congested Sample Input Data:
    lane_id  hour  num_drones  payload_kg  wind_speed  distance_km  \
0        7    14           8         4.5         4.1         1.55   

   temperature  visibility_km  day_of_week  battery_level  drone_speed_kmh  \
0         21.1            8.8            5             58             35.7   

   hub_distance_km  is_peak_hour  is_weekday  
0             2.49             1           0